In [6]:
# ============================================================
# СРАВНЕНИЕ ТЕХНИКИ ИЗ ДАТАСЕТОВ С ТРЕБОВАНИЯМИ
# Источник требований: CSV
# Источник соответствий классов: пользовательский CLASS_MAPPING
# ============================================================

import re
import json
from pathlib import Path
from collections import defaultdict

import pandas as pd
import yaml


# ============================================================
# 1. ПУТИ
# ============================================================

CSV_PATH = Path(
    r"C:\Users\Ева\Desktop\Хакатон\Данные от ЛЦТ\data_by_gen_ai\data_by_gen_ai\equip.csv"
)

DATASETS_ROOT = Path(
    r"C:\Users\Ева\Desktop\Хакатон\data_preproc\datasets"
)

OUTPUT_PATH = Path(
    r"C:\Users\Ева\Desktop\Хакатон\data_preproc\datasets\dataset_info\equipment_requirements_vs_datasets.xlsx"
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. CLASS_MAPPING
# ============================================================
# ВАЖНО:
#
# Здесь НЕ создаём и НЕ изменяем mapping.
#
# Используется твой уже существующий CLASS_MAPPING.
#
# Если запускаешь ноутбук с нуля, вставь сюда СВОЙ mapping
# без каких-либо изменений.
#
# Например:
#
# CLASS_MAPPING = {
#     "Autocran": "Автокран",
#     ...
# }
#
# ============================================================

if "CLASS_MAPPING" not in globals():
    raise RuntimeError(
        "CLASS_MAPPING не найден.\n"
        "Сначала выполни ячейку со своим CLASS_MAPPING."
    )


# ============================================================
# 3. СЛУЖЕБНЫЕ ФУНКЦИИ
# ============================================================

def clean_text(value):
    """Приведение значения к строке без лишних пробелов."""
    if value is None:
        return ""

    if pd.isna(value):
        return ""

    value = str(value).strip()

    # Убираем переводы строк внутри ячейки
    value = re.sub(r"\s+", " ", value)

    return value.strip()


def normalize_equipment_name(value):
    """
    Нормализация ТОЛЬКО для сравнения.

    Исходные названия при этом не изменяются.
    """
    value = clean_text(value)

    if not value:
        return ""

    value = value.lower()

    # ё/е считаем одинаковыми при сравнении
    value = value.replace("ё", "е")

    # разные варианты тире
    value = value.replace("–", "-")
    value = value.replace("—", "-")
    value = value.replace("−", "-")

    # пробелы вокруг дефиса
    value = re.sub(r"\s*-\s*", "-", value)

    # повторные пробелы
    value = re.sub(r"\s+", " ", value)

    return value.strip()


def unique_preserve_order(values):
    """Удаление дублей с сохранением исходного порядка."""
    result = []
    seen = set()

    for value in values:
        value = clean_text(value)

        if not value:
            continue

        key = normalize_equipment_name(value)

        if key not in seen:
            seen.add(key)
            result.append(value)

    return result


# ============================================================
# 4. ЧТЕНИЕ CSV
# ============================================================

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"CSV не найден:\n{CSV_PATH}"
    )


# utf-8-sig хорошо работает с CSV из Excel
try:
    requirements_df = pd.read_csv(
        CSV_PATH,
        sep=",",
        encoding="utf-8-sig"
    )
except UnicodeDecodeError:
    requirements_df = pd.read_csv(
        CSV_PATH,
        sep=",",
        encoding="cp1251"
    )


print("CSV прочитан:")
print(f"  Файл: {CSV_PATH}")
print(f"  Строк: {len(requirements_df)}")
print()
print("Колонки:")
print(list(requirements_df.columns))


# ============================================================
# 5. ПРОВЕРКА КОЛОНОК CSV
# ============================================================

required_columns = [
    "Этап",
    "Название этапа",
    "Обязательная техника",
    "Условная техника",
    "Необязательная/возможная техника",
]

missing_columns = [
    col
    for col in required_columns
    if col not in requirements_df.columns
]

if missing_columns:
    raise ValueError(
        "В CSV отсутствуют необходимые колонки:\n"
        + "\n".join(f"- {x}" for x in missing_columns)
    )


# ============================================================
# 6. РАЗБОР СТРОКИ С ТЕХНИКОЙ
# ============================================================

def parse_equipment_cell(value):
    """
    Разбирает содержимое ячейки CSV.

    Основной разделитель:
        ;

    Специальный случай:

    Геодезический измерительный прибор
    (SURVEY_INSTRUMENT; one_of: Тахеометр, GNSS-приемник, Теодолит)
    — work_023, work_026

    Здесь основной объект должен остаться:
        Геодезический измерительный прибор

    а содержимое one_of будет сохранено отдельно.
    """

    value = clean_text(value)

    if not value:
        return []

    if value.lower() == "нет":
        return []

    result = []

    # --------------------------------------------------------
    # Сначала разделяем по ;
    # --------------------------------------------------------

    parts = [
        clean_text(x)
        for x in value.split(";")
        if clean_text(x)
    ]

    for part in parts:

        # ----------------------------------------------------
        # Убираем служебные идентификаторы вида:
        #
        # — work_023, work_026
        # ----------------------------------------------------

        part = re.sub(
            r"\s*[—–-]\s*work_[A-Za-z0-9_,\s-]+$",
            "",
            part,
            flags=re.IGNORECASE
        ).strip()

        # ----------------------------------------------------
        # Если это конструкция:
        #
        # Техника (CODE; one_of: ...)
        #
        # оставляем основное название до "("
        # ----------------------------------------------------

        if "(" in part:

            main_name = part.split("(", 1)[0].strip()

            if main_name:
                result.append(main_name)

            continue

        # ----------------------------------------------------
        # Если осталась конструкция "one_of:"
        # ----------------------------------------------------

        if re.match(r"^one_of\s*:", part, flags=re.IGNORECASE):
            continue

        result.append(part)

    return unique_preserve_order(result)


# ============================================================
# 7. ОТДЕЛЬНО СОХРАНЯЕМ ONE_OF
# ============================================================

def extract_one_of(value):
    """
    Извлекает варианты из конструкции:

    one_of: Тахеометр, GNSS-приемник, Теодолит
    """

    value = clean_text(value)

    if not value:
        return []

    matches = re.findall(
        r"one_of\s*:\s*([^)]*)",
        value,
        flags=re.IGNORECASE
    )

    result = []

    for match in matches:

        for item in match.split(","):
            item = clean_text(item)

            if item:
                result.append(item)

    return unique_preserve_order(result)


# ============================================================
# 8. СТРОИМ НОРМАЛИЗОВАННЫЙ СПИСОК ТРЕБОВАНИЙ
# ============================================================

# equipment_requirements:
#
# {
#   normalized_name: {
#       "original_names": [...],
#       "categories": set(...),
#       "stages": [...]
#   }
# }

equipment_requirements = defaultdict(
    lambda: {
        "original_names": [],
        "categories": set(),
        "stages": [],
    }
)


# Для аудита каждой исходной записи CSV
requirement_rows = []


category_columns = {
    "Обязательная": "Обязательная техника",
    "Условная": "Условная техника",
    "Необязательная/возможная": "Необязательная/возможная техника",
}


for _, row in requirements_df.iterrows():

    stage_code = clean_text(row["Этап"])
    stage_name = clean_text(row["Название этапа"])

    for category, column in category_columns.items():

        cell = clean_text(row[column])

        if not cell or cell.lower() == "нет":
            continue

        equipment_list = parse_equipment_cell(cell)
        one_of_list = extract_one_of(cell)

        for equipment in equipment_list:

            norm = normalize_equipment_name(equipment)

            if not norm:
                continue

            if equipment not in equipment_requirements[norm]["original_names"]:
                equipment_requirements[norm]["original_names"].append(
                    equipment
                )

            equipment_requirements[norm]["categories"].add(
                category
            )

            stage_record = (
                stage_code,
                stage_name,
                category,
            )

            if stage_record not in equipment_requirements[norm]["stages"]:
                equipment_requirements[norm]["stages"].append(
                    stage_record
                )

            requirement_rows.append({
                "Этап": stage_code,
                "Название этапа": stage_name,
                "Категория": category,
                "Исходная строка CSV": cell,
                "Техника": equipment,
                "Варианты one_of": "; ".join(one_of_list),
            })


print()
print(
    f"Уникальных названий техники в требованиях: "
    f"{len(equipment_requirements)}"
)


# ============================================================
# 9. ТАБЛИЦА ИСХОДНЫХ ТРЕБОВАНИЙ
# ============================================================

requirements_audit_df = pd.DataFrame(
    requirement_rows
)


# ============================================================
# 10. ЧТЕНИЕ YOLO DATASET
# ============================================================

def load_yaml_names(yaml_path):
    """
    Читает names из YOLO data.yaml.

    Поддерживает:

    names:
      - excavator
      - truck

    и:

    names:
      0: excavator
      1: truck
    """

    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    names = data.get("names", {})

    if isinstance(names, list):
        return {
            i: str(name)
            for i, name in enumerate(names)
        }

    if isinstance(names, dict):
        result = {}

        for key, name in names.items():
            result[int(key)] = str(name)

        return result

    return {}


def find_label_dirs(dataset_dir):
    """
    Ищет стандартные YOLO-папки labels.
    """

    candidates = [
        dataset_dir / "labels",
        dataset_dir / "train" / "labels",
        dataset_dir / "valid" / "labels",
        dataset_dir / "val" / "labels",
        dataset_dir / "test" / "labels",
    ]

    return [
        path
        for path in candidates
        if path.exists() and path.is_dir()
    ]


# ============================================================
# 11. СБОР ВСЕХ КЛАССОВ ИЗ DATASET
# ============================================================

dataset_class_rows = []


yaml_files = []

for yaml_path in DATASETS_ROOT.rglob("*.yaml"):

    if yaml_path.name.lower() in {
        "data.yaml",
        "dataset.yaml",
    }:
        yaml_files.append(yaml_path)


print()
print(f"Найдено YOLO YAML: {len(yaml_files)}")


for yaml_path in yaml_files:

    dataset_name = yaml_path.parent.name

    try:
        names = load_yaml_names(yaml_path)
    except Exception as e:
        print(
            f"⚠ Не удалось прочитать {yaml_path}: {e}"
        )
        continue

    if not names:
        continue

    label_dirs = find_label_dirs(yaml_path.parent)

    # количество объектов каждого класса
    object_counts = defaultdict(int)

    # количество файлов разметки каждого класса
    file_counts = defaultdict(int)

    for label_dir in label_dirs:

        for txt_path in label_dir.rglob("*.txt"):

            classes_in_file = set()

            try:
                with open(
                    txt_path,
                    "r",
                    encoding="utf-8"
                ) as f:

                    for line in f:

                        line = line.strip()

                        if not line:
                            continue

                        parts = line.split()

                        try:
                            class_id = int(parts[0])
                        except (ValueError, IndexError):
                            continue

                        object_counts[class_id] += 1
                        classes_in_file.add(class_id)

                for class_id in classes_in_file:
                    file_counts[class_id] += 1

            except Exception:
                continue

    # --------------------------------------------------------
    # Создаём запись для каждого класса YAML
    # --------------------------------------------------------

    for class_id, class_name in names.items():

        class_name = clean_text(class_name)

        dataset_class_rows.append({
            "Dataset": dataset_name,
            "YAML": str(yaml_path),
            "Class ID": class_id,
            "Исходный класс датасета": class_name,
            "Количество объектов": object_counts.get(
                class_id,
                0
            ),
            "Количество файлов": file_counts.get(
                class_id,
                0
            ),
        })


dataset_classes_df = pd.DataFrame(
    dataset_class_rows
)


print(
    f"Найдено записей классов: "
    f"{len(dataset_classes_df)}"
)


# ============================================================
# 12. СОПОСТАВЛЕНИЕ ЧЕРЕЗ ТВОЙ CLASS_MAPPING
# ============================================================

audit_rows = []


for _, row in dataset_classes_df.iterrows():

    dataset_class = clean_text(
        row["Исходный класс датасета"]
    )

    dataset_class_norm = normalize_equipment_name(
        dataset_class
    )

    # --------------------------------------------------------
    # Ищем ТОЛЬКО в пользовательском CLASS_MAPPING
    # --------------------------------------------------------

    mapped_name = CLASS_MAPPING.get(dataset_class)

    mapping_found = mapped_name is not None

    if mapping_found:
        mapped_name = clean_text(mapped_name)
    else:
        mapped_name = ""

    # --------------------------------------------------------
    # Нормализуем результат mapping ТОЛЬКО для сравнения
    # --------------------------------------------------------

    mapped_norm = normalize_equipment_name(
        mapped_name
    )

    # --------------------------------------------------------
    # Ищем mapped_name среди требований CSV
    # --------------------------------------------------------

    requirement_match = ""

    if mapped_norm in equipment_requirements:
        originals = equipment_requirements[
            mapped_norm
        ]["original_names"]

        if originals:
            requirement_match = originals[0]

    # --------------------------------------------------------
    # Категории требований
    # --------------------------------------------------------

    categories = ""

    if mapped_norm in equipment_requirements:

        categories = "; ".join(
            sorted(
                equipment_requirements[
                    mapped_norm
                ]["categories"]
            )
        )

    # --------------------------------------------------------
    # Статус
    # --------------------------------------------------------

    if not mapping_found:

        status = "Не сопоставлен в CLASS_MAPPING"

    elif requirement_match:

        status = "Есть в требованиях и в датасетах"

    else:

        status = "Есть в датасетах, нет в требованиях"

    audit_rows.append({

        "Dataset": row["Dataset"],

        "Исходный класс датасета":
            dataset_class,

        "Количество объектов":
            row["Количество объектов"],

        "Количество файлов":
            row["Количество файлов"],

        "Есть в CLASS_MAPPING":
            "Да" if mapping_found else "Нет",

        "Твой CLASS_MAPPING":
            mapped_name,

        "Название в требованиях CSV":
            requirement_match,

        "Категория требования":
            categories,

        "Статус":
            status,
    })


audit_df = pd.DataFrame(audit_rows)


# ============================================================
# 13. СОЗДАЁМ СПИСКИ ТЕХНИКИ ИЗ DATASET
# ============================================================

dataset_equipment = {}

for _, row in audit_df.iterrows():

    mapped_name = clean_text(
        row["Твой CLASS_MAPPING"]
    )

    if not mapped_name:
        continue

    norm = normalize_equipment_name(
        mapped_name
    )

    if norm not in dataset_equipment:

        dataset_equipment[norm] = {
            "name": mapped_name,
            "datasets": [],
            "objects": 0,
        }

    dataset_equipment[norm]["objects"] += int(
        row["Количество объектов"]
    )

    dataset_equipment[norm]["datasets"].append(
        row["Dataset"]
    )


# ============================================================
# 14. СТРОИМ СВОДНУЮ ТАБЛИЦУ
# ============================================================

summary_rows = []


# ------------------------------------------------------------
# ГРУППА 1
# Есть и в требованиях, и в датасетах
# ------------------------------------------------------------

for norm, info in dataset_equipment.items():

    if norm not in equipment_requirements:
        continue

    req = equipment_requirements[norm]

    summary_rows.append({

        "Статус":
            "Есть в требованиях и в датасетах",

        "Техника":
            req["original_names"][0],

        "Категория требования":
            "; ".join(sorted(req["categories"])),

        "Датасеты":
            "; ".join(
                sorted(set(info["datasets"]))
            ),

        "Количество объектов":
            info["objects"],

        "Этапы":
            "; ".join(
                sorted(
                    set(
                        stage_name
                        for _, stage_name, _ in req["stages"]
                    )
                )
            ),
    })


# ------------------------------------------------------------
# ГРУППА 2
# Есть в датасетах, но нет в требованиях
# ------------------------------------------------------------

for norm, info in dataset_equipment.items():

    if norm in equipment_requirements:
        continue

    summary_rows.append({

        "Статус":
            "Есть в датасетах, нет в требованиях",

        "Техника":
            info["name"],

        "Категория требования":
            "",

        "Датасеты":
            "; ".join(
                sorted(set(info["datasets"]))
            ),

        "Количество объектов":
            info["objects"],

        "Этапы":
            "",
    })


# ------------------------------------------------------------
# ГРУППА 3
# Есть в требованиях, но нет в датасетах
# ------------------------------------------------------------

for norm, req in equipment_requirements.items():

    if norm in dataset_equipment:
        continue

    summary_rows.append({

        "Статус":
            "Есть в требованиях, нет в датасетах",

        "Техника":
            req["original_names"][0],

        "Категория требования":
            "; ".join(
                sorted(req["categories"])
            ),

        "Датасеты":
            "",

        "Количество объектов":
            0,

        "Этапы":
            "; ".join(
                sorted(
                    set(
                        stage_name
                        for _, stage_name, _ in req["stages"]
                    )
                )
            ),
    })


summary_df = pd.DataFrame(summary_rows)


# ============================================================
# 15. ПОРЯДОК СОРТИРОВКИ
# ============================================================

status_order = {
    "Есть в требованиях и в датасетах": 1,
    "Есть в датасетах, нет в требованиях": 2,
    "Есть в требованиях, нет в датасетах": 3,
}


summary_df["_Порядок"] = summary_df["Статус"].map(
    status_order
)

summary_df = (
    summary_df
    .sort_values(
        ["_Порядок", "Техника"],
        kind="stable"
    )
    .drop(columns="_Порядок")
    .reset_index(drop=True)
)


# ============================================================
# 16. НЕСОПОСТАВЛЕННЫЕ КЛАССЫ
# ============================================================

unmatched_df = audit_df[
    audit_df["Статус"] == "Не сопоставлен в CLASS_MAPPING"
].copy()


# ============================================================
# 17. ПОКРЫТИЕ ТРЕБОВАНИЙ
# ============================================================

coverage_rows = []

for norm, req in equipment_requirements.items():

    in_dataset = norm in dataset_equipment

    dataset_info = dataset_equipment.get(norm)

    coverage_rows.append({

        "Техника":
            req["original_names"][0],

        "Категория требования":
            "; ".join(
                sorted(req["categories"])
            ),

        "Есть в датасетах":
            "Да" if in_dataset else "Нет",

        "Количество объектов":
            dataset_info["objects"]
            if dataset_info
            else 0,

        "Датасеты":
            "; ".join(
                sorted(
                    set(dataset_info["datasets"])
                )
            )
            if dataset_info
            else "",

        "Этапы":
            "; ".join(
                sorted(
                    set(
                        stage_name
                        for _, stage_name, _
                        in req["stages"]
                    )
                )
            ),
    })


coverage_df = pd.DataFrame(
    coverage_rows
)


# ============================================================
# 18. ПОЛНАЯ ТАБЛИЦА DATASET-КЛАССОВ
# ============================================================

dataset_classes_with_mapping_df = audit_df.copy()


# ============================================================
# 19. СОХРАНЕНИЕ В EXCEL
# ============================================================

with pd.ExcelWriter(
    OUTPUT_PATH,
    engine="openpyxl"
) as writer:

    # Главная таблица
    summary_df.to_excel(
        writer,
        sheet_name="Сводная таблица",
        index=False
    )

    # Самое важное — аудит mapping
    dataset_classes_with_mapping_df.to_excel(
        writer,
        sheet_name="Аудит соответствий",
        index=False
    )

    # Все исходные требования CSV
    requirements_audit_df.to_excel(
        writer,
        sheet_name="Исходные требования CSV",
        index=False
    )

    # Покрытие требований
    coverage_df.to_excel(
        writer,
        sheet_name="Покрытие требований",
        index=False
    )

    # Только классы, которые не попали в mapping
    unmatched_df.to_excel(
        writer,
        sheet_name="Не сопоставлены",
        index=False
    )

    # Все исходные классы из YAML
    dataset_classes_df.to_excel(
        writer,
        sheet_name="Классы датасетов",
        index=False
    )


# ============================================================
# 20. ФОРМАТИРОВАНИЕ EXCEL
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter


wb = load_workbook(OUTPUT_PATH)


for ws in wb.worksheets:

    # Закрепляем первую строку
    ws.freeze_panes = "A2"

    # Автофильтр
    if ws.max_row > 1:
        ws.auto_filter.ref = ws.dimensions

    # Заголовки
    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    # Все ячейки
    for row in ws.iter_rows():

        for cell in row:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )

    # Ширина столбцов
    for column_cells in ws.columns:

        max_length = 0

        for cell in column_cells:

            value = str(
                cell.value
            ) if cell.value is not None else ""

            max_length = max(
                max_length,
                len(value)
            )

        width = min(
            max(max_length + 2, 12),
            60
        )

        column_letter = get_column_letter(
            column_cells[0].column
        )

        ws.column_dimensions[
            column_letter
        ].width = width


# ============================================================
# 21. ЦВЕТОВАЯ ПОДСВЕТКА СТАТУСОВ
# ============================================================

ws = wb["Сводная таблица"]

for row in range(2, ws.max_row + 1):

    status_cell = ws.cell(
        row=row,
        column=1
    )

    status = status_cell.value

    if status == "Есть в требованиях и в датасетах":

        fill = PatternFill(
            fill_type="solid",
            fgColor="C6EFCE"
        )

    elif status == "Есть в датасетах, нет в требованиях":

        fill = PatternFill(
            fill_type="solid",
            fgColor="FFEB9C"
        )

    elif status == "Есть в требованиях, нет в датасетах":

        fill = PatternFill(
            fill_type="solid",
            fgColor="FFC7CE"
        )

    else:
        continue

    for cell in ws[row]:
        cell.fill = fill


wb.save(OUTPUT_PATH)


# ============================================================
# 22. КРАТКАЯ СТАТИСТИКА
# ============================================================

print()
print("=" * 70)
print("ГОТОВО")
print("=" * 70)

print()
print(f"Файл сохранён:")
print(OUTPUT_PATH)

print()
print("СТАТИСТИКА:")

print(
    f"  Уникальных требований: "
    f"{len(equipment_requirements)}"
)

print(
    f"  Уникальных mapped-названий из датасетов: "
    f"{len(dataset_equipment)}"
)

print()

for status, count in summary_df["Статус"].value_counts().items():

    print(
        f"  {status}: {count}"
    )

print()

print(
    f"  Не сопоставлено через CLASS_MAPPING: "
    f"{len(unmatched_df)}"
)

print()
print("=" * 70)

CSV прочитан:
  Файл: C:\Users\Ева\Desktop\Хакатон\Данные от ЛЦТ\data_by_gen_ai\data_by_gen_ai\equip.csv
  Строк: 37

Колонки:
['Этап', 'Название этапа', 'Обязательная техника', 'Условная техника', 'Необязательная/возможная техника']

Уникальных названий техники в требованиях: 140

Найдено YOLO YAML: 10
Найдено записей классов: 70

ГОТОВО

Файл сохранён:
C:\Users\Ева\Desktop\Хакатон\data_preproc\datasets\dataset_info\equipment_requirements_vs_datasets.xlsx

СТАТИСТИКА:
  Уникальных требований: 140
  Уникальных mapped-названий из датасетов: 26

  Есть в требованиях, нет в датасетах: 127
  Есть в требованиях и в датасетах: 13
  Есть в датасетах, нет в требованиях: 13

  Не сопоставлено через CLASS_MAPPING: 1

